# 第 5 课 · 把一轮训练变成递归反馈

**本课用时约 40 分钟。** 对比冻结、一次训练、多轮均匀训练和按弱项调整训练分布。

## 学完后你能做到

- 对照相同训练预算下的多种策略。
- 读出弱项如何影响下一轮练习分配。
- 用多个随机种子报告变化范围，而不是挑单次高分。

## 跟着操作

运行固定种子的完整比较，再用自己的种子列表复做。

运行代码格时，把光标放在格子里，按 **Shift + Enter**。看到输出后再运行下一格。若第一次误点或修改了上方变量，选 Jupyter 菜单里的 **Kernel → Restart Kernel and Run All Cells**，从第一格重新运行。本课不要求预先懂强化学习，也不需要 GPU。

## 每轮有两条反馈

训练数据是否通过校验会改变**解题策略**。这一轮哪种指令经常失败，会改变**下一轮抽什么题**。均匀课程只更新解题策略，固定下一轮练习比例。自适应课程根据各组平滑失败率分配下一轮样本。

自适应权重为 `(失败数 + 1) / (尝试数 + 2)`。平滑项使没有尝试的组仍然有正权重，避免整组永久消失。模型仍必须能抽到正确解题规则。

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from rsi_lab import INITIAL, NAMES, evaluate, train

model, trace = train(seed=19, adaptive=True, budget=32, rounds=5)
for row in trace:
    print("轮", row["epoch"], "尝试", row["attempts"], "通过", row["kept"])
    print("开发集各组:", [round(v, 2) for v in row["dev"]["by_task"]])

先逐行看五轮记录。若一组的平滑失败率高于其他组，它下一轮被抽中的概率就更高，实际抽到的次数仍会波动；若正样本改变了策略，该组后续可能更容易答对。多轮均匀对照可以单独检验「中途更新策略」的作用，自适应对照再检验「按失败调整数据」的额外作用。

In [ ]:
settings = [
    ("冻结", False, 0, 0),
    # 160 × 1 和 32 × 5 使用相同预算，差异来自何时更新策略。
    ("一次更新", False, 160, 1),
    ("多轮均匀", False, 32, 5),
    ("多轮自适应", True, 32, 5),
]
for seed in (19, 20, 21):
    for label, adaptive, budget, rounds in settings:
        weights, _ = train(seed, adaptive, budget, rounds)
        score = evaluate(weights, range(75, 100))
        print(seed, label, "期望", f"{score['expected']:.1%}",
              "贪心", f"{score['greedy']:.1%}",
              "最弱组", f"{min(score['by_task']):.1%}")

从**同一个种子的一组对照**开始解释。多轮均匀与一次更新使用相同总样本预算，差别在更新频率；多轮自适应与多轮均匀差在课程调整。再横向比较种子，判断观察是否稳健。这个有限、确定性答案空间里，均匀和自适应可能非常接近，不应强行宣称自适应一定更好。

In [ ]:
from rsi_lab import next_mix

attempts, kept = [10, 10, 10, 10], [8, 5, 2, 0]
mix = next_mix(attempts, kept)
print(dict(zip(NAMES, [round(x / sum(mix), 3) for x in mix])))
assert mix[-1] > mix[0]

## 自检

- [ ] 多轮训练的总采样预算为 160。
- [ ] 每轮改进使用固定开发集测量。
- [ ] 我能解释两条反馈路径分别改变什么。
- [ ] 我会用种子配对的对照结果，而不拿不同种子直接作因果比较。

第六课会做反事实实验、检查奖励漏洞，并整理一份可复现报告。

[返回课程首页并选择下一节](recursive_self_improvement.ipynb)